# TT-16 — Decision Tree Regressor
## Định giá cước chuyến xe — bảng giá dạng LUẬT cho tổng đài

Notebook này đi theo đúng 12 bước ở mục 5 của README:

1. Tải 1 tháng dữ liệu, lấy mẫu 200.000 dòng
2. Làm sạch dữ liệu (4 bước) + ghi lại số dòng bị loại
3. Tạo đặc trưng thời gian
4. EDA
5. Baseline (DummyRegressor + công thức tuyến tính thủ công)
6. Cây không giới hạn độ sâu → chứng minh overfit
7. MAE train/test theo max_depth = 1..20
8. Vẽ hàm bậc thang
9. Cây max_depth=5 → export_text + vẽ cây
10. Bảng tra cước (CSV)
11. Kiểm tra % chuyến sai số trong ±15%
12. So sánh Random Forest & Linear Regression

**Lưu ý:** mỗi cell chỉ làm đúng một việc, chạy tuần tự từ trên xuống. Bạn tự chạy (Shift+Enter) từng cell.

## 0. Cài đặt & import thư viện

Nếu thiếu thư viện nào, bỏ dấu `#` ở dòng `!pip install ...` bên dưới rồi chạy trước.

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib pyarrow requests tqdm

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)


## Bước 1 — Tải dữ liệu & lấy mẫu 200.000 dòng

Dữ liệu chính thức: **NYC Yellow Taxi Trip Records** (định dạng `.parquet`, cập nhật hàng tháng).
Trang chủ: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

File được lưu trên CloudFront với URL dạng:
`https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_YYYY-MM.parquet`

Cell dưới sẽ **tự tải file nếu chưa có**, chọn tháng qua biến `YEAR_MONTH`.
Mỗi file khoảng 45–50MB, ~3 triệu dòng — tải xong sẽ lấy mẫu ngẫu nhiên 200.000 dòng ngay để nhẹ máy.

> ⚠️ Cell này CẦN kết nối Internet khi bạn tự chạy. Nếu mạng của bạn chặn CloudFront,
> đổi `YEAR_MONTH` sang tháng khác, hoặc tải thủ công file `.parquet` từ trang TLC ở trên
> rồi để cùng thư mục với notebook, cell sẽ tự nhận ra và bỏ qua bước tải.

In [ ]:
YEAR_MONTH = "2024-01"           # đổi tháng ở đây nếu cần, định dạng YYYY-MM
DATA_DIR = "data"
FILE_NAME = f"yellow_tripdata_{YEAR_MONTH}.parquet"
FILE_PATH = os.path.join(DATA_DIR, FILE_NAME)
DOWNLOAD_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{FILE_NAME}"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(FILE_PATH):
    import requests
    from tqdm import tqdm

    print(f"Đang tải {DOWNLOAD_URL} ...")
    resp = requests.get(DOWNLOAD_URL, stream=True, timeout=60)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(FILE_PATH, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    print("Tải xong:", FILE_PATH)
else:
    print("File đã tồn tại, bỏ qua bước tải:", FILE_PATH)


In [ ]:
# Đọc parquet rồi lấy mẫu ngẫu nhiên 200.000 dòng (3 triệu dòng/tháng quá nặng để luyện tập)
SAMPLE_SIZE = 200_000

df_full = pd.read_parquet(FILE_PATH)
print("Tổng số dòng gốc:", len(df_full))

df = df_full.sample(n=min(SAMPLE_SIZE, len(df_full)), random_state=RANDOM_STATE).reset_index(drop=True)
del df_full  # giải phóng bộ nhớ
print("Số dòng sau khi lấy mẫu:", len(df))
df.head()


## Bước 2 — Làm sạch dữ liệu (4 bước bắt buộc)

Theo README:
1. Lấy mẫu 200.000 dòng — đã làm ở bước 1
2. Loại chuyến vô lý: `fare_amount <= 0`, `trip_distance <= 0` hoặc `> 100 miles`, `passenger_count == 0`
3. ⚠️ Không dùng `tip_amount`, `tolls_amount`, `total_amount` làm đặc trưng (rò rỉ dữ liệu)
4. Tạo đặc trưng thời gian — làm ở bước 3

Ghi lại rõ số dòng bị loại ở MỖI điều kiện để đưa vào báo cáo.

In [ ]:
n_before = len(df)
log_loai_bo = {}

# (a) fare_amount <= 0 -> chuyến huỷ / hoàn tiền
mask_fare = df["fare_amount"] <= 0
log_loai_bo["fare_amount <= 0"] = int(mask_fare.sum())

# (b) trip_distance vô lý
mask_dist = (df["trip_distance"] <= 0) | (df["trip_distance"] > 100)
log_loai_bo["trip_distance <= 0 hoac > 100 miles"] = int(mask_dist.sum())

# (c) passenger_count == 0 (hoặc thiếu)
mask_pax = df["passenger_count"].fillna(0) == 0
log_loai_bo["passenger_count == 0"] = int(mask_pax.sum())

mask_loai = mask_fare | mask_dist | mask_pax
df_clean = df.loc[~mask_loai].copy()

print("--- SỐ DÒNG BỊ LOẠI THEO TỪNG ĐIỀU KIỆN ---")
for k, v in log_loai_bo.items():
    print(f"  {k:40s}: {v:>8,} dòng")
print(f"  {'TỔNG LOẠI (không trùng lặp)':40s}: {n_before - len(df_clean):>8,} dòng")
print(f"\nSố dòng trước khi làm sạch : {n_before:,}")
print(f"Số dòng sau khi làm sạch   : {len(df_clean):,}  ({len(df_clean)/n_before:.1%} giữ lại)")

df = df_clean
del df_clean


In [ ]:
# ⚠️ RÒ RỈ DỮ LIỆU: các cột này chỉ biết SAU khi chuyến đi kết thúc,
# tuyệt đối không dùng làm đặc trưng khi dự đoán fare_amount
COT_RO_RI = ["tip_amount", "tolls_amount", "total_amount"]
print("Các cột KHÔNG được dùng làm đặc trưng (rò rỉ):", COT_RO_RI)


## Bước 3 — Tạo đặc trưng thời gian (giờ, thứ, cao điểm)

In [ ]:
df["gio_don"] = df["tpep_pickup_datetime"].dt.hour
df["thu_trong_tuan"] = df["tpep_pickup_datetime"].dt.dayofweek  # 0 = Thứ Hai ... 6 = Chủ Nhật

# Giờ cao điểm: 7-9h sáng và 16-19h chiều các ngày trong tuần (giả định nghiệp vụ phổ biến)
def la_gio_cao_diem(row):
    if row["thu_trong_tuan"] >= 5:  # cuối tuần: không tính cao điểm giờ công sở
        return 0
    return int((7 <= row["gio_don"] <= 9) or (16 <= row["gio_don"] <= 19))

df["gio_cao_diem"] = df.apply(la_gio_cao_diem, axis=1)

df[["tpep_pickup_datetime", "gio_don", "thu_trong_tuan", "gio_cao_diem"]].head()


## Bước 4 — EDA: quãng đường vs cước, giá trung bình theo giờ

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Scatter quãng đường vs cước (lấy mẫu nhỏ hơn để vẽ cho nhẹ)
sample_plot = df.sample(n=min(5000, len(df)), random_state=RANDOM_STATE)
ax[0].scatter(sample_plot["trip_distance"], sample_plot["fare_amount"], s=5, alpha=0.3)
ax[0].set_xlabel("Quãng đường (miles)")
ax[0].set_ylabel("Cước (USD)")
ax[0].set_title("Quãng đường vs Cước")

# Giá trung bình theo giờ trong ngày
gia_theo_gio = df.groupby("gio_don")["fare_amount"].mean()
ax[1].bar(gia_theo_gio.index, gia_theo_gio.values, color="steelblue")
ax[1].set_xlabel("Giờ trong ngày")
ax[1].set_ylabel("Cước trung bình (USD)")
ax[1].set_title("Cước trung bình theo giờ trong ngày")

plt.tight_layout()
plt.show()


## Chuẩn bị dữ liệu train/test

Đặc trưng sử dụng (KHÔNG dùng cột rò rỉ): `trip_distance`, `passenger_count`,
`PULocationID`, `DOLocationID`, `gio_don`, `thu_trong_tuan`, `gio_cao_diem`.

In [ ]:
FEATURES = [
    "trip_distance", "passenger_count",
    "PULocationID", "DOLocationID",
    "gio_don", "thu_trong_tuan", "gio_cao_diem",
]
TARGET = "fare_amount"

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)


## Bước 5 — Baseline

- `DummyRegressor(strategy="mean")`: luôn đoán giá trung bình
- Công thức tuyến tính thủ công: `12.000 VNĐ + 8.500 VNĐ × km` (quy đổi tương đương theo USD để so trên cùng thang đo dữ liệu NYC)

Đây là "sàn" tối thiểu — mọi mô hình cây sau này phải đánh bại baseline này mới có ý nghĩa.

In [ ]:
def bao_cao_metric(ten, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{ten:35s} | MAE={mae:6.2f} | MAPE={mape:6.2f}% | RMSE={rmse:6.2f}")
    return {"model": ten, "MAE": mae, "MAPE": mape, "RMSE": rmse}

ket_qua = []

# Baseline 1: DummyRegressor (mean)
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
ket_qua.append(bao_cao_metric("Baseline - DummyRegressor(mean)", y_test, dummy.predict(X_test)))

# Baseline 2: công thức tuyến tính thủ công (quy đổi hệ số cho thang đo USD của NYC)
# 12.000 VND ~ hệ số chặn, 8.500 VND/km ~ hệ số góc -> quy đổi tỉ lệ tương ứng ra USD/mile
# Ở đây minh hoạ bằng chính công thức gốc, đơn vị tính là "đơn vị tiền tệ giả định"
HE_SO_CHAN = 12000 / 25000   # giả sử 1 USD ~ 25.000 VND để đưa về thang so sánh
HE_SO_GOC = 8500 / 25000 * 1.60934  # đổi km -> mile rồi quy đổi VND -> USD

y_pred_thu_cong = HE_SO_CHAN + HE_SO_GOC * X_test["trip_distance"]
ket_qua.append(bao_cao_metric("Baseline - Cong thuc tuyen tinh thu cong", y_test, y_pred_thu_cong))


## Bước 6 — Cây KHÔNG giới hạn độ sâu → chứng minh overfit

So sánh MAE trên tập train vs tập test khi để cây phát triển tự do (`max_depth=None`).
MAE train sẽ gần bằng 0 (cây học thuộc lòng), còn MAE test sẽ tệ hơn nhiều → đây chính là overfit.

In [ ]:
tree_full = DecisionTreeRegressor(random_state=RANDOM_STATE)  # max_depth=None mặc định
tree_full.fit(X_train, y_train)

mae_train_full = mean_absolute_error(y_train, tree_full.predict(X_train))
mae_test_full = mean_absolute_error(y_test, tree_full.predict(X_test))

print(f"Số lá của cây không giới hạn độ sâu: {tree_full.get_n_leaves():,}")
print(f"Độ sâu thực tế của cây             : {tree_full.get_depth()}")
print(f"MAE train : {mae_train_full:.3f}")
print(f"MAE test  : {mae_test_full:.3f}")
print(f"\n=> Chênh lệch train/test rất lớn => cây học thuộc lòng dữ liệu train => OVERFIT rõ ràng.")


## Bước 7 — MAE train/test theo max_depth = 1..20 → chọn điểm tối ưu

In [ ]:
depths = range(1, 21)
mae_train_list, mae_test_list = [], []

for d in depths:
    t = DecisionTreeRegressor(max_depth=d, min_samples_leaf=500, random_state=RANDOM_STATE)
    t.fit(X_train, y_train)
    mae_train_list.append(mean_absolute_error(y_train, t.predict(X_train)))
    mae_test_list.append(mean_absolute_error(y_test, t.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(list(depths), mae_train_list, marker="o", label="MAE - Train")
plt.plot(list(depths), mae_test_list, marker="o", label="MAE - Test")
plt.xlabel("max_depth")
plt.ylabel("MAE (USD)")
plt.title("MAE Train/Test theo độ sâu cây")
plt.xticks(list(depths))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_depth = list(depths)[int(np.argmin(mae_test_list))]
print(f"Độ sâu cho MAE test thấp nhất trong khoảng khảo sát: max_depth = {best_depth}")
print("Lưu ý: mục tiêu là bảng tra CƯỚC cho tổng đài (yêu cầu ①), nên vẫn ưu tiên cây nông (4-5 tầng) dù không phải điểm MAE thấp tuyệt đối.")


## Bước 8 — ⭐ Vẽ hàm dự đoán theo quãng đường → nhìn thấy hình BẬC THANG

Cố định các đặc trưng khác theo giá trị phổ biến nhất, chỉ cho `trip_distance` chạy từ 0 đến giá trị lớn,
so sánh với dữ liệu thật để thấy rõ dạng bậc thang của cây hồi quy (khác với đường mượt của hồi quy tuyến tính).

In [ ]:
tree5_demo = DecisionTreeRegressor(max_depth=5, min_samples_leaf=500, random_state=RANDOM_STATE)
tree5_demo.fit(X_train, y_train)

gia_tri_pho_bien = {
    "passenger_count": X_train["passenger_count"].mode()[0],
    "PULocationID": X_train["PULocationID"].mode()[0],
    "DOLocationID": X_train["DOLocationID"].mode()[0],
    "gio_don": 14,
    "thu_trong_tuan": 2,
    "gio_cao_diem": 0,
}

khoang_cach_gia = np.linspace(0.1, X_train["trip_distance"].quantile(0.99), 300)
X_bac_thang = pd.DataFrame({
    "trip_distance": khoang_cach_gia,
    **{k: [v] * len(khoang_cach_gia) for k, v in gia_tri_pho_bien.items()},
})[FEATURES]

y_bac_thang = tree5_demo.predict(X_bac_thang)

plt.figure(figsize=(9, 5))
plt.scatter(df["trip_distance"], df["fare_amount"], s=3, alpha=0.15, label="Dữ liệu thật", color="gray")
plt.plot(khoang_cach_gia, y_bac_thang, color="crimson", linewidth=2, label="Dự đoán của cây (max_depth=5)")
plt.xlim(0, X_train["trip_distance"].quantile(0.99))
plt.xlabel("Quãng đường (miles)")
plt.ylabel("Cước (USD)")
plt.title("Hàm dự đoán dạng BẬC THANG của Decision Tree Regressor")
plt.legend()
plt.show()

print(f"Số mức giá (số lá) mà cây max_depth=5 có thể đưa ra: {tree5_demo.get_n_leaves()}")


## Bước 9 — Cây `max_depth=5` chính thức → `export_text` + vẽ cây

Đây là cây "sản phẩm" — đủ nông để tổng đài viên tra tay, theo đúng yêu cầu nghiệp vụ ①.

In [ ]:
tree = DecisionTreeRegressor(
    max_depth=5,               # đủ nông để in thành bảng tra
    min_samples_leaf=500,      # mỗi mức giá phải dựa trên >=500 chuyến thật
    criterion="squared_error",
    random_state=RANDOM_STATE,
)
tree.fit(X_train, y_train)

print("Số lá:", tree.get_n_leaves())
print()
print(export_text(tree, feature_names=FEATURES))


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    tree,
    feature_names=FEATURES,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=5,
)
plt.title("Cây quyết định định giá cước (max_depth=5)")
plt.savefig("cay_quyet_dinh.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu hình: cay_quyet_dinh.png")


In [ ]:
y_pred_test = tree.predict(X_test)
ket_qua.append(bao_cao_metric("Decision Tree (max_depth=5)", y_test, y_pred_test))

for r in ket_qua:
    print(r)


## Bước 10 — ⭐ Chuyển cây thành BẢNG TRA CƯỚC (CSV) cho tổng đài

Mỗi lá của cây tương ứng với một "luật" (một dải điều kiện) và một mức giá cố định.
Trích xuất trực tiếp từ cấu trúc `tree_.feature` / `tree_.threshold` để đảm bảo bảng tra
khớp 100% với cây đã huấn luyện, đọc được ngay bằng một chuỗi so sánh if/else.

In [ ]:
def trich_xuat_luat(tree_model, feature_names):
    """Duyệt cây và trả về danh sách luật dạng: [(dieu_kien_text, gia_du_doan, so_mau)]"""
    tree_ = tree_model.tree_
    feat_name = [
        feature_names[i] if i != -2 else "leaf"
        for i in tree_.feature
    ]
    luat_list = []

    def duyet(node, dieu_kien):
        if tree_.feature[node] != -2:  # không phải lá
            ten_dt = feat_name[node]
            nguong = tree_.threshold[node]
            duyet(tree_.children_left[node], dieu_kien + [f"{ten_dt} <= {nguong:.2f}"])
            duyet(tree_.children_right[node], dieu_kien + [f"{ten_dt} > {nguong:.2f}"])
        else:
            gia = tree_.value[node][0][0]
            so_mau = tree_.n_node_samples[node]
            luat_list.append({
                "dieu_kien": " VA ".join(dieu_kien) if dieu_kien else "(mọi trường hợp)",
                "gia_du_doan_usd": round(float(gia), 2),
                "so_mau_lich_su": int(so_mau),
            })

    duyet(0, [])
    return luat_list

bang_tra_cuoc = pd.DataFrame(trich_xuat_luat(tree, FEATURES))
bang_tra_cuoc = bang_tra_cuoc.sort_values("gia_du_doan_usd").reset_index(drop=True)

os.makedirs("outputs", exist_ok=True)
bang_tra_cuoc.to_csv("outputs/bang_tra_cuoc.csv", index=False, encoding="utf-8-sig")

print(f"Đã xuất {len(bang_tra_cuoc)} luật (= {tree.get_n_leaves()} lá) ra outputs/bang_tra_cuoc.csv")
bang_tra_cuoc


## Bước 11 — Kiểm tra: bao nhiêu % chuyến có sai số trong ±15%?

In [ ]:
sai_so_phan_tram = np.abs((y_pred_test - y_test) / y_test) * 100
ty_le_dat_15pct = (sai_so_phan_tram <= 15).mean() * 100

print(f"Tỉ lệ chuyến có sai số dự đoán trong khoảng ±15%: {ty_le_dat_15pct:.1f}%")
print(f"Yêu cầu nghiệp vụ ③ là sai số chấp nhận ±15% -> ", end="")
print("ĐẠT" if ty_le_dat_15pct >= 80 else "CHƯA ĐẠT, cần xem lại độ sâu cây hoặc đặc trưng")

plt.figure(figsize=(7, 4))
plt.hist(sai_so_phan_tram.clip(0, 100), bins=40, color="teal", alpha=0.8)
plt.axvline(15, color="red", linestyle="--", label="Ngưỡng ±15%")
plt.xlabel("Sai số tuyệt đối (%)")
plt.ylabel("Số chuyến")
plt.title("Phân phối sai số dự đoán cước (%)")
plt.legend()
plt.show()


## Bước 12 — So sánh với Random Forest Regressor (TT-17) và Linear Regression

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_leaf=50,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
ket_qua.append(bao_cao_metric("Random Forest Regressor", y_test, rf.predict(X_test)))

lr = LinearRegression()
lr.fit(X_train, y_train)
ket_qua.append(bao_cao_metric("Linear Regression", y_test, lr.predict(X_test)))

bang_so_sanh = pd.DataFrame(ket_qua)
bang_so_sanh


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(bang_so_sanh["model"], bang_so_sanh["MAE"], color="slateblue")
plt.ylabel("MAE (USD)")
plt.title("So sánh MAE giữa các mô hình")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("mae_theo_depth.png", dpi=150, bbox_inches="tight")
plt.show()


## VÌ SAO CÂY KHÔNG NGOẠI SUY ĐƯỢC?

Decision Tree Regressor chỉ có thể trả về **giá trị trung bình của các điểm dữ liệu train đã rơi vào từng lá**.
Cây không có công thức đại số (như `y = a + bx`) để "ngoại suy" ra ngoài phạm vi dữ liệu đã thấy —
nó chỉ biết so sánh ngưỡng (`trip_distance <= x?`) rồi đi theo nhánh có sẵn.

**Ví dụ cụ thể:** một chuyến đi 200 km (xa hơn bất kỳ chuyến nào trong dữ liệu train) sẽ vẫn rơi vào
nhánh "quãng đường > ngưỡng lớn nhất" và nhận **đúng mức giá trung bình của lá xa nhất** mà cây từng thấy —
dù thực tế chuyến 200 km phải đắt hơn nhiều. Cell dưới đây minh hoạ trực tiếp điều này.

In [ ]:
chuyen_test = pd.DataFrame([{
    "trip_distance": 200,   # rất xa, ngoài phạm vi dữ liệu train
    "passenger_count": 1,
    "PULocationID": int(X_train["PULocationID"].mode()[0]),
    "DOLocationID": int(X_train["DOLocationID"].mode()[0]),
    "gio_don": 14,
    "thu_trong_tuan": 2,
    "gio_cao_diem": 0,
}])[FEATURES]

gia_du_doan_200km = tree.predict(chuyen_test)[0]
quang_duong_xa_nhat_trong_train = X_train["trip_distance"].max()

print(f"Quãng đường xa nhất từng thấy trong tập train: {quang_duong_xa_nhat_trong_train:.1f} miles")
print(f"Giá cây dự đoán cho chuyến 200 miles          : {gia_du_doan_200km:.2f} USD")
print("=> Cây KHÔNG thể ước lượng giá tăng thêm theo quãng đường vượt ngoài dữ liệu đã học,")
print("   nó chỉ lặp lại đúng mức giá của lá xa nhất mà nó từng gặp trong lúc huấn luyện.")


## Tổng kết & sản phẩm bàn giao

- `outputs/bang_tra_cuoc.csv` — bảng tra cước dạng luật cho tổng đài (⭐ sản phẩm chính)
- `cay_quyet_dinh.png` — hình vẽ cây quyết định
- `mae_theo_depth.png` — biểu đồ so sánh MAE giữa các mô hình

Nhớ đối chiếu lại với mục 6 (TIÊU CHÍ HOÀN THÀNH) và mục 7 (CẠM BẪY) trong README trước khi nộp bài.